## Імпорт необхідних бібліотек

In [1]:
%matplotlib tk

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, CheckButtons
import scipy.signal as signal

Формула чистої гармоніки:
$$y(t) = A * sin(\omega * t + \varphi)$$

In [2]:
# Масив часу
t = np.linspace(0, 10, 1000)

def generate_harmonic(t, amplitude, frequency, phase):
    """Генерує чисту гармоніку"""
    return amplitude * np.sin(frequency * t + phase)

def generate_noise(t, noise_mean, noise_covariance):
    """Генерує шум із нормальним розподілом"""
    std_dev = np.sqrt(noise_covariance)
    return np.random.normal(noise_mean, std_dev, len(t))

def harmonic_with_noise(t, amplitude, frequency, phase, noise_mean, noise_covariance, show_noise, noise_array=None):
    """Повертає чисту гармоніку, шум і підсумковий сигнал"""
    clean_harmonic = generate_harmonic(t, amplitude, frequency, phase)

    if noise_array is None:
        noise_array = generate_noise(t, noise_mean, noise_covariance)

    if show_noise:
        signal_with_noise = clean_harmonic + noise_array
    else:
        signal_with_noise = clean_harmonic

    return signal_with_noise, clean_harmonic, noise_array

## Графік

У програмі реалізовано:
- головне вікно з одним полем графіка;
- повзунки для керування амплітудою, частотою, фазою та параметрами шуму;
- чекбокс для увімкнення або вимкнення відображення шуму;
- логіку кешування шуму: якщо змінюються параметри гармоніки, шум не генерується наново; якщо змінюються параметри шуму - створюється новий шум;
- фільтр низьких частот на основі `scipy.signal.iirfilter`;
- відображення на одному графіку:
  - чистої гармоніки,
  - зашумленої гармоніки,
  - відфільтрованої гармоніки;
- кнопку `Reset`, яка повертає всі параметри до початкових значень і відновлює початковий масив шуму.

In [ ]:
# Початкові значення параметрів
init_amplitude = 1.0
init_frequency = 1.0
init_phase = 0.0
init_noise_mean = 0.0
init_noise_covariance = 0.1
init_show_noise = True
init_cutoff = 5.0  # Початкова частота зрізу

# Вікно
fig, ax = plt.subplots(figsize=(10, 8))
plt.subplots_adjust(bottom=0.55)

# Пам'ять шуму
initial_noise = generate_noise(t, init_noise_mean, init_noise_covariance)
current_noise = np.copy(initial_noise)
prev_noise_mean = init_noise_mean
prev_noise_covariance = init_noise_covariance

# Генерація початкових сигналів
signal_with_noise, clean_harmonic, _ = harmonic_with_noise(
    t,
    init_amplitude,
    init_frequency,
    init_phase,
    init_noise_mean,
    init_noise_covariance,
    init_show_noise,
    current_noise
)

# Фільтрація
def apply_filter(noisy_signal, cutoff_freq):
    """Фільтр Баттерворта низьких частот"""
    fs = 100.0  # 1000 точок / 10 секунд
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_freq / nyquist
    if normal_cutoff <= 0:
        normal_cutoff = 0.01
    if normal_cutoff >= 1:
        normal_cutoff = 0.99

    b, a = signal.iirfilter(3, normal_cutoff, btype='lowpass', ftype='butter')
    return signal.filtfilt(b, a, noisy_signal)
filtered_signal = apply_filter(signal_with_noise, init_cutoff)

line_noise, = ax.plot(
    t, signal_with_noise,
    label='Зашумлена',
    color='orange',
    alpha=0.8
)

line_clean, = ax.plot(
    t, clean_harmonic,
    label='Чиста',
    color='blue',
    linestyle='--',
    linewidth=2
)

line_filtered, = ax.plot(
    t, filtered_signal,
    label='Відфільтрована',
    color='red',
    linewidth=2
)

ax.set_title('Гармоніка з накладеним шумом та фільтрацією')
ax.set_xlabel('Час (t)')
ax.set_ylabel('Амплітуда (y)')
ax.legend(loc='upper right')
ax.grid(True, linestyle=':', alpha=0.7)

# Віджети
ax_amp = plt.axes([0.15, 0.45, 0.65, 0.03])
ax_freq = plt.axes([0.15, 0.40, 0.65, 0.03])
ax_phase = plt.axes([0.15, 0.35, 0.65, 0.03])
ax_nmean = plt.axes([0.15, 0.30, 0.65, 0.03])
ax_ncov = plt.axes([0.15, 0.25, 0.65, 0.03])
ax_cutoff = plt.axes([0.15, 0.20, 0.65, 0.03])
ax_reset = plt.axes([0.15, 0.10, 0.1, 0.05])
ax_check = plt.axes([0.85, 0.25, 0.1, 0.1])

# Слайдери
slider_amp = Slider(ax_amp, 'Amplitude', 0.1, 5.0, valinit=init_amplitude)
slider_freq = Slider(ax_freq, 'Frequency', 0.1, 10.0, valinit=init_frequency)
slider_phase = Slider(ax_phase, 'Phase', 0.0, 2*np.pi, valinit=init_phase)
slider_noise_mean = Slider(ax_nmean, 'Noise Mean', -2.0, 2.0, valinit=init_noise_mean)
slider_noise_cov = Slider(ax_ncov, 'Noise Covariance', 0.0, 1.0, valinit=init_noise_covariance)
slider_cutoff = Slider(ax_cutoff, 'Cutoff Frequency', 0.1, 49.0, valinit=init_cutoff)

# Кнопка і чекбокс
button_reset = Button(ax_reset, 'Reset', hovercolor='0.975')
check_noise = CheckButtons(ax_check, ['Show Noise'], [init_show_noise])

# Оновлення/скидання
def update(val):
    global current_noise, prev_noise_mean, prev_noise_covariance

    amp = slider_amp.val
    freq = slider_freq.val
    phase = slider_phase.val
    n_mean = slider_noise_mean.val
    n_cov = slider_noise_cov.val
    cutoff = slider_cutoff.val
    show_n = check_noise.get_status()[0]

    # Якщо змінились лише параметри шуму — генеруємо новий шум
    if n_mean != prev_noise_mean or n_cov != prev_noise_covariance:
        current_noise = generate_noise(t, n_mean, n_cov)
        prev_noise_mean = n_mean
        prev_noise_covariance = n_cov

    # Формуємо сигнал
    y_signal, y_clean, _ = harmonic_with_noise(
        t, amp, freq, phase, n_mean, n_cov, show_n, current_noise
    )

    noisy_for_filter = generate_harmonic(t, amp, freq, phase) + current_noise
    y_filtered = apply_filter(noisy_for_filter, cutoff)

    # Оновлення ліній
    line_clean.set_ydata(y_clean)
    line_filtered.set_ydata(y_filtered)

    if show_n:
        line_noise.set_ydata(noisy_for_filter)
        line_noise.set_visible(True)
    else:
        line_noise.set_visible(False)

    # Масштаб осі Y
    y_candidates = [y_clean, y_filtered]
    if show_n:
        y_candidates.append(noisy_for_filter)

    y_min = min(arr.min() for arr in y_candidates) - 0.5
    y_max = max(arr.max() for arr in y_candidates) + 0.5
    ax.set_ylim(y_min, y_max)

    fig.canvas.draw_idle()

# Скидає всі повзунки до початкового стану
def reset_params(event):
    global current_noise, prev_noise_mean, prev_noise_covariance

    current_noise = np.copy(initial_noise)
    prev_noise_mean = init_noise_mean
    prev_noise_covariance = init_noise_covariance

    if check_noise.get_status()[0] != init_show_noise:
        check_noise.set_active(0)

    slider_amp.reset()
    slider_freq.reset()
    slider_phase.reset()
    slider_noise_mean.reset()
    slider_noise_cov.reset()
    slider_cutoff.reset()

    update(None)

# Прив'язка подій
slider_amp.on_changed(update)
slider_freq.on_changed(update)
slider_phase.on_changed(update)
slider_noise_mean.on_changed(update)
slider_noise_cov.on_changed(update)
slider_cutoff.on_changed(update)
check_noise.on_clicked(update)
button_reset.on_clicked(reset_params)

plt.show()